In [1]:
from WindPy import w
import numpy as np
import pandas as pd
import datetime
import os
import json
from tqdm import tqdm

w.start()
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [2]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)

# Load Local Database

In [21]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)
        
len(INDEX_DATA)

69

# Update Existing Tickers

In [4]:
today = "2025-02-28"

for ticker in tqdm(INDEX_DATA):
    data = INDEX_DATA[ticker]
    start = (data.index[-1] + pd.Timedelta(days = 1)).strftime("%Y-%m-%d")
    
    new_data = w.wsd(ticker, 'pe_ttm,dividendyield2', start, today, '', usedf = True)[1]
    
    if new_data.columns[0] == 'OUTMESSAGE':
        print(ticker)
        continue
    
    INDEX_DATA[ticker] = pd.concat([data, new_data])
    INDEX_DATA[ticker].dropna(inplace = True)

100%|██████████████████████████████████████████████████████████████████████████████████| 69/69 [00:38<00:00,  1.77it/s]


# Download New Tickers

In [ ]:
for ticker, start in tqdm(INDEX_START.items()):
    if ticker in INDEX_DATA:
        continue
        
    assert ticker not in INDEX_DATA
    
    data = w.wsd(ticker, 'pe_ttm,dividendyield2', start, today, '', usedf = True)[1]
    
    junk = data['PE_TTM']
    
    data.dropna(inplace = True)
    
    INDEX_DATA[ticker] = data.copy(deep = True)

In [22]:
len(INDEX_DATA)

69

In [25]:
INDEX_DATA['000300.SH'].tail()

,PE_TTM,DIVIDENDYIELD2,CLOSE
2025-02-24,12.7566,3.3624,3969.7169
2025-02-25,12.6115,3.3984,3925.6505
2025-02-26,12.6946,3.3726,3959.9413
2025-02-27,12.7363,3.3613,3968.1159
2025-02-28,12.5543,3.4203,3890.0487


In [26]:
for ticker, data in INDEX_DATA.items():
    path = os.path.join("Data", f"{ticker}.csv")
    data.to_csv(path, index = True, encoding = "utf-8")

In [24]:
for ticker in tqdm(INDEX_DATA):
    start = INDEX_DATA[ticker].index[0]
    end = INDEX_DATA[ticker].index[-1]
    
    if 'CLOSE' in INDEX_DATA[ticker].columns:
        continue
    
    data = w.wsd(ticker, 'close', start, end, 'PriceAdj=B', usedf = True)[1]
    junk = data['CLOSE']
    
    INDEX_DATA[ticker]['CLOSE'] = data['CLOSE']

100%|██████████████████████████████████████████████████████████████████████████████████| 69/69 [00:40<00:00,  1.69it/s]


In [15]:
for ticker in INDEX_DATA:
    if len(INDEX_DATA[ticker].columns) != 3:
        print(ticker)

AttributeError: 'Series' object has no attribute 'columns'

In [13]:
data

,CLOSE
2025-02-20,3928.8959
2025-02-21,3978.4440
2025-02-24,3969.7169
2025-02-25,3925.6505
2025-02-26,3959.9413
2025-02-27,3968.1159
2025-02-28,3890.0487
